# Re-Embed Vectorstores for Cloud Deployment

This notebook reads chunks from the **existing** ChromaDB collections (embedded with `qwen3-embedding:4b` via Ollama)
and re-embeds them using **Jina AI's `jina-embeddings-v3`** so the deploy backend can query
without needing a local Ollama instance.

**Run once before Docker build.** Output goes to `deploy/data/vectorstores/`.

In [ ]:
# Cell 1: Install dependencies (restart kernel after running)
!pip install chromadb openai python-dotenv

In [ ]:
# Cell 2: Configuration
import os
from pathlib import Path
from dotenv import load_dotenv

# Find .env in deploy/ regardless of where kernel started
_cwd = Path(os.getcwd()).resolve()
_candidates = [
    _cwd / ".env",
    _cwd / "deploy" / ".env",
    _cwd.parent / "deploy" / ".env",
]
for _p in _candidates:
    if _p.exists():
        load_dotenv(_p, override=True)
        print(f"Loaded .env from: {_p}")
        break
else:
    print("WARNING: .env not found — tried:", [str(p) for p in _candidates])

JINA_API_KEY = os.getenv("EMBEDDING_API_KEY", "")
JINA_BASE_URL = "https://api.jina.ai/v1"
EMBEDDING_MODEL = "jina-embeddings-v3"

# Find project root (directory that contains "Knwlodge base")
_root = _cwd
for _check in [_cwd, _cwd.parent, _cwd.parent.parent]:
    if (_check / "Knwlodge base").exists():
        _root = _check
        break
PROJECT_ROOT = _root
SOURCE_VS_DIR = PROJECT_ROOT / "Knwlodge base" / "vectorstores"
DEST_VS_DIR = PROJECT_ROOT / "deploy" / "data" / "vectorstores"

COLLECTIONS = {
    1: "barcelona_cat1_local_regulations",
    2: "barcelona_cat2_safety_building",
    3: "barcelona_cat3_social_value",
    4: "barcelona_cat4_heritage",
    5: "barcelona_cat5_mobility",
}

assert JINA_API_KEY, "EMBEDDING_API_KEY not set in .env!"
print(f"Source: {SOURCE_VS_DIR}")
print(f"Dest:   {DEST_VS_DIR}")
print(f"Model:  {EMBEDDING_MODEL}")
print(f"Jina key loaded: ...{JINA_API_KEY[-8:]}")

In [ ]:
# Cell 3: Embedding function using Jina AI (OpenAI-compatible)
from openai import OpenAI
import time

client = OpenAI(
    api_key=JINA_API_KEY,
    base_url=JINA_BASE_URL,
)


def embed_batch(texts: list[str], batch_size: int = 20) -> list[list[float]]:
    """Embed texts in small batches via Jina AI. Respects 100K tokens/min limit."""
    all_embeddings = []
    total = len(texts)
    for i in range(0, total, batch_size):
        batch = texts[i : i + batch_size]
        retries = 0
        while retries < 5:
            try:
                response = client.embeddings.create(
                    model=EMBEDDING_MODEL,
                    input=batch,
                )
                all_embeddings.extend([item.embedding for item in response.data])
                break
            except Exception as e:
                if "429" in str(e) or "rate" in str(e).lower():
                    wait = 30 * (retries + 1)
                    print(f"  Rate limited at {i}/{total}, waiting {wait}s...")
                    time.sleep(wait)
                    retries += 1
                else:
                    raise
        else:
            raise RuntimeError(f"Failed after 5 retries at batch {i}")

        done = min(i + batch_size, total)
        if done % 200 == 0 or done == total:
            print(f"  Embedded {done}/{total}")

        # Small delay between batches to stay under rate limit
        time.sleep(1)

    return all_embeddings


# Quick test
test = embed_batch(["hello world"])
dim = len(test[0])
print(f"Embedding dim: {dim}")
print("Jina AI connection works!")

In [ ]:
# Cell 4: Extract all chunks from existing collections
import chromadb


def extract_all_chunks(category: int) -> dict:
    """Read ALL chunks (ids, documents, metadatas) from an existing collection."""
    db_path = SOURCE_VS_DIR / f"chroma_cat{category}"
    if not db_path.exists():
        print(f"  WARNING: {db_path} not found")
        return {"ids": [], "documents": [], "metadatas": []}

    client = chromadb.PersistentClient(path=str(db_path))
    collection = client.get_collection(name=COLLECTIONS[category])
    count = collection.count()
    print(f"  Cat {category}: {count} chunks")

    # ChromaDB get() with no filter returns all documents
    # Fetch in pages to avoid memory issues
    all_ids, all_docs, all_metas = [], [], []
    page_size = 5000
    offset = 0
    while offset < count:
        result = collection.get(
            limit=page_size,
            offset=offset,
            include=["documents", "metadatas"],
        )
        all_ids.extend(result["ids"])
        all_docs.extend(result["documents"])
        all_metas.extend(result["metadatas"])
        offset += page_size

    return {"ids": all_ids, "documents": all_docs, "metadatas": all_metas}


# Extract all categories
all_data = {}
for cat in COLLECTIONS:
    all_data[cat] = extract_all_chunks(cat)

total = sum(len(d["ids"]) for d in all_data.values())
print(f"\nTotal chunks to re-embed: {total}")

In [ ]:
# Cell 5: Re-embed and write to new ChromaDB collections
import shutil


for cat, col_name in COLLECTIONS.items():
    data = all_data[cat]
    if not data["ids"]:
        print(f"Cat {cat}: no data, skipping")
        continue

    print(f"\n{'='*60}")
    print(f"Category {cat}: {col_name}")
    print(f"Chunks: {len(data['ids'])}")
    print(f"{'='*60}")

    # 1. Embed all documents
    print("Embedding...")
    embeddings = embed_batch(data["documents"])
    print(f"  Got {len(embeddings)} embeddings of dim {len(embeddings[0])}")

    # 2. Create new ChromaDB collection
    dest_path = DEST_VS_DIR / f"chroma_cat{cat}"
    if dest_path.exists():
        shutil.rmtree(dest_path)
    dest_path.mkdir(parents=True, exist_ok=True)

    new_client = chromadb.PersistentClient(path=str(dest_path))
    new_collection = new_client.create_collection(
        name=col_name,
        metadata={"hnsw:space": "cosine"},
    )

    # 3. Add in batches (ChromaDB has a batch size limit)
    batch_size = 5000
    for i in range(0, len(data["ids"]), batch_size):
        end = min(i + batch_size, len(data["ids"]))
        new_collection.add(
            ids=data["ids"][i:end],
            documents=data["documents"][i:end],
            metadatas=data["metadatas"][i:end],
            embeddings=embeddings[i:end],
        )
        print(f"  Added batch {i}-{end}")

    print(f"  Done! New collection has {new_collection.count()} chunks")

print("\n" + "="*60)
print("ALL CATEGORIES RE-EMBEDDED SUCCESSFULLY")
print("="*60)

In [ ]:
# Cell 6: Verify — query each new collection
test_query = "solar panels on rooftops in Eixample"
print(f"Test query: '{test_query}'\n")

# Get query embedding from Jina
query_embedding = embed_batch([test_query])[0]

for cat, col_name in COLLECTIONS.items():
    dest_path = DEST_VS_DIR / f"chroma_cat{cat}"
    if not dest_path.exists():
        print(f"Cat {cat}: not found")
        continue

    vc = chromadb.PersistentClient(path=str(dest_path))
    col = vc.get_collection(name=col_name)

    results = col.query(query_embeddings=[query_embedding], n_results=2)
    docs = results["documents"][0]
    dists = results["distances"][0]

    print(f"Cat {cat} ({col.count()} chunks):")
    for j, (doc, dist) in enumerate(zip(docs, dists)):
        print(f"  [{j+1}] dist={dist:.4f} | {doc[:100]}...")
    print()

In [ ]:
# Cell 7: Print final sizes

print("New vectorstore sizes:")
for cat in COLLECTIONS:
    dest_path = DEST_VS_DIR / f"chroma_cat{cat}"
    if dest_path.exists():
        total_size = sum(f.stat().st_size for f in dest_path.rglob("*") if f.is_file())
        print(f"  chroma_cat{cat}: {total_size / 1024 / 1024:.1f} MB")

print("\nReady for Docker build!")
print("Next: cd deploy && docker build -t bcn-api . && docker run -p 8001:8001 --env-file .env bcn-api")